In [1]:
load_ext jupyter_black

In [2]:
from copy import deepcopy
import numpy as np
import itertools
import pandas as pd
from tqdm import tqdm

In [3]:
demographics = {
    "prism": [
        "age",
        "gender",
        "employment_status",
        "education",
        "marital_status",
        "english_proficiency",
        "religion",
        "ethnicity",
        "birth_region",
        "reside_region",
        "lm_familiarity",
    ],
    "chen": [
        "Gender",
        "human_Gender",
    ],
    "cad_en": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_fr": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_pt": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_it": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
}

domains = ["legal", "salary", "medical", "benefits", "political"]

In [4]:
class WelfordVariance:
    # https://en.wikipedia.org/wiki/Algorithms_for_calculating_variance#Welford's_online_algorithm
    def __init__(self):  # Comparison to ShiftDataVariance:
        self.mean = 0.0  # = K + Ex / n
        self.count = 0  # = n
        self.M2 = 0.0  # = Ex2 - (Ex)^2 / n

    def add_variable(self, x: float):
        self.count += 1
        old_mean = self.mean
        self.mean += (x - self.mean) / self.count
        self.M2 += (x - old_mean) * (x - self.mean)

    def remove_variable(self, x: float):
        self.count -= 1
        new_mean = self.mean
        self.mean -= (x - self.mean) / self.count
        self.M2 -= (x - new_mean) * (x - self.mean)

    def get_mean(self) -> float:
        return self.mean

    def get_variance(self) -> float:
        return self.M2 / self.count

    def get_sample_variance(self) -> float:
        return self.M2 / (self.count - 1)

In [5]:
from scipy.stats import f as f_dist


def one_way_anova(*groups: tuple[int, float, float]) -> dict:
    """
    Perform a one-way ANOVA from group summary statistics.

    Parameters
    ----------
    *groups : tuples of (n, mean, variance)
        Each tuple describes one group:
          - n   (int)   : number of observations
          - mean (float): sample mean
          - var  (float): sample variance (unbiased, ddof=1)

    Returns
    -------
    dict with keys:
        F        - F-statistic
        p_value  - p-value
        df_between - degrees of freedom between groups
        df_within  - degrees of freedom within groups
        SS_between - sum of squares between groups
        SS_within  - sum of squares within groups
        MS_between - mean square between groups
        MS_within  - mean square within groups
    """
    if len(groups) < 2:
        raise ValueError("At least two groups are required.")

    ns, means, variances = zip(*groups)

    for i, (n, mean, var) in enumerate(groups):
        if n < 2:
            raise ValueError(f"Group {i+1} must have at least 2 samples (n={n}).")
        if var < 0:
            raise ValueError(f"Group {i+1} has a negative variance ({var}).")

    k = len(groups)  # number of groups
    N = sum(ns)  # total observations
    grand_mean = sum(n * m for n, m in zip(ns, means)) / N

    # Between-group sum of squares
    SS_between = sum(n * (m - grand_mean) ** 2 for n, m in zip(ns, means))

    # Within-group sum of squares  (sum of (n-1)*var for each group)
    SS_within = sum((n - 1) * v for n, v in zip(ns, variances))

    df_between = k - 1
    df_within = N - k

    MS_between = SS_between / df_between
    MS_within = SS_within / df_within

    F = MS_between / MS_within
    p_value = f_dist.sf(F, df_between, df_within)  # survival function = 1 - CDF

    return {
        "F": F,
        "p_value": p_value,
        "df_between": df_between,
        "df_within": df_within,
        "SS_between": SS_between,
        "SS_within": SS_within,
        "MS_between": MS_between,
        "MS_within": MS_within,
    }


def print_anova_table(results: dict, alpha: float = 0.05) -> None:
    print("\n" + "=" * 52)
    print("         ONE-WAY ANOVA TABLE")
    print("=" * 52)
    print(f"{'Source':<12} {'SS':>10} {'df':>6} {'MS':>10}")
    print("-" * 52)
    print(
        f"{'Between':<12} {results['SS_between']:>10.4f} {results['df_between']:>6} {results['MS_between']:>10.4f}"
    )
    print(
        f"{'Within':<12} {results['SS_within']:>10.4f} {results['df_within']:>6} {results['MS_within']:>10.4f}"
    )
    print("-" * 52)
    print(f"\n  F-statistic : {results['F']:.4f}")
    print(f"  p-value     : {results['p_value']:.4f}")
    decision = "Reject H₀" if results["p_value"] < alpha else "Fail to reject H₀"
    print(f"  Decision    : {decision}  (α = {alpha})")
    print("=" * 52 + "\n")

In [6]:
questions = pd.read_pickle("data/Llama-3.1-8B-Instruct_questions.gz")
questions_correct_answers = dict(zip(questions.q_id, questions.correct_answer))
domain_qid_map = {
    domain: questions.loc[questions["domain"] == domain, "q_id"].tolist()
    for domain in domains
}

In [7]:
for dataset in [
    "prism",
]:
    all_cols = deepcopy(demographics[dataset])
    df = pd.read_pickle(f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_answers.gz")
    df = df.rename(columns={"label": "Gender"})

    for c in [qid for d in domains for qid in domain_qid_map[d] if d != "salary"]:
        df[c] = 1 * (df[c].str.lower() == questions_correct_answers[c])

    for c in domain_qid_map["salary"]:
        df[c] = df[c].str.replace(",", "").str.extract(r"^[^\d]*(\d+)").astype(float)

    for domain in domains:
        df[domain] = df[[qid for qid in domain_qid_map[domain]]].mean(axis=1)
        if domain != "salary":
            df[domain] = df[domain] * 100

    df = df.drop(
        columns=[f"q_{i}" for i in range(50)]
        + ["q_59", "q_60"]
        + [f"q_{i}" for i in range(61, 211)]
    )

    df_linguistic = pd.read_pickle(
        f"data/{dataset + '_utterances' if dataset != 'chen' else dataset}_linguistic.gz"
    ).drop(
        columns=[
            "s_neutral_model_response",
            "s_neutral_user_prompt",
            "model_response_liwc_Segment",
            "user_prompt_liwc_Segment",
        ],
        errors="ignore",
    )
    for c in ["politeness_user_prompt", "politeness_model_response"]:
        if c in df_linguistic:
            df_linguistic[c] = df_linguistic[c].replace(
                {"impolite": 0, "neutral": 0.5, "polite": 1, "somewhat polite": 0.75}
            )
    df_linguistic = df_linguistic.rename(columns={"gpt_description": "topic"})
    all_cols += ["topic"]
    group_cols = ["conversation_id"] + all_cols
    df_linguistic = (
        df_linguistic.groupby(group_cols)[
            [
                c
                for c in df_linguistic.columns
                if ("model_response" in c or "user_prompt" in c)
                and (c not in ["model_response", "user_prompt"])
            ]
        ]
        .mean()
        .reset_index()
    )
    df = df.merge(
        df_linguistic[
            ["conversation_id"]
            + [
                c
                for c in df_linguistic.columns
                if "model_response" in c or "user_prompt" in c or c == "topic"
            ]
        ],
        on="conversation_id",
    )

    df = df.loc[df["topic"] != "Outliers"]
    for domain in tqdm(domains):
        results = {}
        for column in demographics[dataset]:
            results[column] = {}
            filtered_df = df.loc[
                ~(df[column].isna())
                & (df[column] != "Prefer not to say")
                & (df[column] != "Other")
                & (df[column] != "Unknown")
            ]
            for group in filtered_df[column].unique():
                results[column][group] = {}
                group_df = filtered_df.loc[filtered_df[column] == group]
                for topic in group_df["topic"].unique():
                    results[column][group][topic] = group_df.loc[
                        group_df["topic"] == topic, domain
                    ]

        same_group_same_topic = WelfordVariance()
        same_group_diff_topic = WelfordVariance()
        diff_group_same_topic = WelfordVariance()
        diff_group_diff_topic = WelfordVariance()

        for column in tqdm(results):
            for group1 in results[column]:
                for group2 in results[column]:
                    for topic1 in results[column][group1]:
                        for topic2 in results[column][group2]:
                            if group1 == group2 and topic1 == topic2:
                                var = same_group_same_topic
                            elif group1 != group2 and topic1 == topic2:
                                var = diff_group_same_topic
                            elif group1 == group2 and topic1 != topic2:
                                var = same_group_diff_topic
                            else:
                                var = diff_group_diff_topic
                            for e in itertools.product(
                                results[column][group1][topic1],
                                results[column][group2][topic2],
                            ):
                                var.add_variable(abs(e[1] - e[0]))

        print(same_group_same_topic.count)
        print(
            domain,
            same_group_same_topic.get_mean(),
            diff_group_same_topic.get_mean(),
            same_group_diff_topic.get_mean(),
            diff_group_diff_topic.get_mean(),
        )
        print_anova_table(
            one_way_anova(
                *[
                    (
                        same_group_same_topic.count,
                        same_group_same_topic.get_mean(),
                        same_group_same_topic.get_variance(),
                    ),
                    (
                        diff_group_same_topic.count,
                        diff_group_same_topic.get_mean(),
                        diff_group_same_topic.get_variance(),
                    ),
                    (
                        same_group_diff_topic.count,
                        same_group_diff_topic.get_mean(),
                        same_group_diff_topic.get_variance(),
                    ),
                    (
                        diff_group_diff_topic.count,
                        diff_group_diff_topic.get_mean(),
                        diff_group_diff_topic.get_variance(),
                    ),
                ]
            ),
            alpha=0.05,
        )

/tmp/ipykernel_7736/302451165.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[domain] = df[[qid for qid in domain_qid_map[domain]]].mean(axis=1)
/tmp/ipykernel_7736/302451165.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[domain] = df[[qid for qid in domain_qid_map[domain]]].mean(axis=1)
/tmp/ipykernel_7736/302451165.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using p

8401685
legal 3.583054113549784 3.6034045158601824 4.244922643579192 4.2243370715079935

         ONE-WAY ANOVA TABLE
Source               SS     df         MS
----------------------------------------------------
Between      8285218.5273      3 2761739.5091
Within       3619981168.9268 315312423    11.4806
----------------------------------------------------

  F-statistic : 240556.7144
  p-value     : 0.0000
  Decision    : Reject H₀  (α = 0.05)




%|                                                                                                                                                                                                                | 0/11 [00:00<?, ?it/s]
%|██████████████████▏                                                                                                                                                                                     | 1/11 [00:06<01:06,  6.65s/it]
%|████████████████████████████████████▎                                                                                                                                                                   | 2/11 [00:13<00:58,  6.54s/it]
%|██████████████████████████████████████████████████████▌                                                                                                                                                 | 3/11 [00:19<00:52,  6.58s/it]
%|█████████████████████████████████████████████████████████████

8401685
salary 1681.7219117355196 1699.5011171586941 1874.4886299618968 1879.6450199066303

         ONE-WAY ANOVA TABLE
Source               SS     df         MS
----------------------------------------------------
Between      700548563083.8928      3 233516187694.6310
Within       703993116209511.7500 315312423 2232684.3627
----------------------------------------------------

  F-statistic : 104589.8792
  p-value     : 0.0000
  Decision    : Reject H₀  (α = 0.05)




%|                                                                                                                                                                                                                | 0/11 [00:00<?, ?it/s]
%|██████████████████▏                                                                                                                                                                                     | 1/11 [00:06<01:04,  6.43s/it]
%|████████████████████████████████████▎                                                                                                                                                                   | 2/11 [00:12<00:58,  6.49s/it]
%|██████████████████████████████████████████████████████▌                                                                                                                                                 | 3/11 [00:19<00:52,  6.52s/it]
%|█████████████████████████████████████████████████████████████

8401685
medical 8.444787920517825 8.554087863372816 10.118750313080538 10.133981701577845

         ONE-WAY ANOVA TABLE
Source               SS     df         MS
----------------------------------------------------
Between      53267720.0767      3 17755906.6922
Within       18254774386.1251 315312423    57.8942
----------------------------------------------------

  F-statistic : 306695.5440
  p-value     : 0.0000
  Decision    : Reject H₀  (α = 0.05)




%|                                                                                                                                                                                                                | 0/11 [00:00<?, ?it/s]
%|██████████████████▏                                                                                                                                                                                     | 1/11 [00:06<01:06,  6.63s/it]
%|████████████████████████████████████▎                                                                                                                                                                   | 2/11 [00:13<00:59,  6.56s/it]
%|██████████████████████████████████████████████████████▌                                                                                                                                                 | 3/11 [00:19<00:52,  6.57s/it]
%|█████████████████████████████████████████████████████████████

8401685
benefits 9.255073476333191 9.330688758286806 10.514884074519392 10.45348736556509

         ONE-WAY ANOVA TABLE
Source               SS     df         MS
----------------------------------------------------
Between      28421100.9786      3 9473700.3262
Within       22915503326.2774 315312423    72.6755
----------------------------------------------------

  F-statistic : 130356.0896
  p-value     : 0.0000
  Decision    : Reject H₀  (α = 0.05)




%|                                                                                                                                                                                                                | 0/11 [00:00<?, ?it/s]
%|██████████████████▏                                                                                                                                                                                     | 1/11 [00:06<01:05,  6.56s/it]
%|████████████████████████████████████▎                                                                                                                                                                   | 2/11 [00:13<00:58,  6.54s/it]
%|██████████████████████████████████████████████████████▌                                                                                                                                                 | 3/11 [00:19<00:52,  6.55s/it]
%|█████████████████████████████████████████████████████████████

8401685
political 3.7768095328498665 3.8169420109807843 4.277747763672296 4.278877627414052

         ONE-WAY ANOVA TABLE
Source               SS     df         MS
----------------------------------------------------
Between      4640731.1152      3 1546910.3717
Within       3600682346.4154 315312423    11.4194
----------------------------------------------------

  F-statistic : 135463.2291
  p-value     : 0.0000
  Decision    : Reject H₀  (α = 0.05)

